[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# SQL Expressions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made that the tasks use: `show_sql`, and the statements of `record_grades`. Run it first.
The tasks do not depend on one another, except that task 5 removes what task 4 adds, and the last
cell removes the scratch folder.


In [1]:
import shutil
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        and_, bindparam, create_engine, delete, event, func, insert, or_, select, update)
from sqlalchemy.dialects.sqlite import insert as sqlite_insert
from sqlalchemy.exc import IntegrityError
from sqlalchemy.pool import StaticPool
from sqlalchemy.dialects import postgresql

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)

def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}


def show_sql(statement, dialect):
    """Print the SQL a statement becomes for one database, and the values that travel beside it."""
    compiled = statement.compile(dialect=dialect)
    for line in str(compiled).splitlines():
        print("   ", line.rstrip())
    print("    values:", compiled.params)

engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SECTION_OF = (                                   # course code -> section, for one term
    select(courses.c.code, sections.c.id)
    .select_from(sections)
    .join(courses)
    .join(terms)
    .where(terms.c.name == bindparam("term"))
)
GRADE = (
    update(enrollments)
    .where(enrollments.c.student_id == bindparam("student"), enrollments.c.section_id == bindparam("section"))
    .values(grade=bindparam("letter"), status="completed")
)
STILL_OPEN = (
    select(func.count())
    .select_from(enrollments.join(sections).join(terms))
    .where(terms.c.name == bindparam("term"), enrollments.c.grade.is_(None))
)


def record_grades(engine, term, sheet):
    """Record a term's grade sheet of (student, course code, grade), and count what was graded and what is still open."""
    with engine.begin() as conn:
        section_of = dict(conn.execute(SECTION_OF, {"term": term}).all())
        rows = [{"student": student, "section": section_of[code], "letter": letter} for student, code, letter in sheet]
        graded = conn.execute(GRADE, rows).rowcount
        return graded, conn.execute(STILL_OPEN, {"term": term}).scalar_one()


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


**1.** Two conditions inside one `.where()`.


In [2]:
four_credit_math = (
    select(courses.c.code, courses.c.title)
    .where(courses.c.department == "Mathematics", courses.c.credits == 4)
    .order_by(courses.c.code)
)
show_sql(four_credit_math, engine.dialect)
with engine.connect() as conn:
    print(conn.execute(four_credit_math).all())


    SELECT courses.code, courses.title
    FROM courses
    WHERE courses.department = ? AND courses.credits = ? ORDER BY courses.code
    values: {'department_1': 'Mathematics', 'credits_1': 4}
[('MAT-120', 'Calculus I'), ('MAT-121', 'Calculus II')]


Two arguments, joined by `AND`, and the SQL shows both, which is the check worth making whenever a
query returns more rows than expected.


**2.** A name with an apostrophe.


In [3]:
with engine.connect() as conn:
    print(conn.execute(select(students.c.name).where(students.c.name.contains("'"))).scalars().all())


["Aoife O'Brien"]


`contains` writes `LIKE '%' || ? || '%'` with the apostrophe as a bound value, so it needed no
escaping at all.


**3.** `between` and `in_` together.


In [4]:
low_marks = (
    select(func.count())
    .select_from(enrollments)
    .where(enrollments.c.section_id.between(21, 30), enrollments.c.grade.in_(["D", "F"]))
)
with engine.connect() as conn:
    print(conn.execute(low_marks).scalar_one(), "enrollments in Fall 2025 ended with a D or an F")


17 enrollments in Fall 2025 ended with a D or an F


**4.** A course and its section, each id from `returning()`.


In [5]:
with engine.begin() as conn:
    art = conn.execute(
        insert(courses).values(code="ART-100", title="Drawing", department="Art", credits=3).returning(courses.c.id)
    ).scalar_one()
    art_section = conn.execute(
        insert(sections).values(course_id=art, term_id=4, capacity=20).returning(sections.c.id)
    ).scalar_one()
print("course", art, "| section", art_section)


course 11 | section 41


The course's id went straight into the section's `course_id`, with no query in between, and both
rows were committed together when the block ended.


**5.** Two deletes, in an order the foreign keys accept.


In [6]:
with engine.begin() as conn:
    print("sections deleted:", conn.execute(delete(sections).where(sections.c.id == art_section)).rowcount)
    print("courses deleted: ", conn.execute(delete(courses).where(courses.c.id == art)).rowcount)


sections deleted: 1
courses deleted:  1


The section goes first, since its foreign key refers to the course. Deleting the course first would
have been refused with `FOREIGN KEY constraint failed`, because `college_engine` switches foreign
keys on.


**6.** One statement, two dialects.


In [7]:
print("SQLite:")
show_sql(GRADE, engine.dialect)
print("PostgreSQL:")
show_sql(GRADE, postgresql.dialect())


SQLite:
    UPDATE enrollments SET status=?, grade=? WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: {'status': 'completed', 'letter': None, 'student': None, 'section': None}
PostgreSQL:
    UPDATE enrollments SET status=%(status)s, grade=%(letter)s WHERE enrollments.student_id = %(student)s AND enrollments.section_id = %(section)s
    values: {'status': 'completed', 'letter': None, 'student': None, 'section': None}


SQLite's driver takes a `?` for every value, and PostgreSQL's takes `%(name)s`, with the names of
the `bindparam`s, `letter`, `student` and `section`, in the SQL itself. The status that `.values()`
fixed became a bind parameter named after its column, with its value, `'completed'`. The others are
`None`, because a `bindparam` has no value until the statement runs.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [SQL Expressions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/06-sql-expressions.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
